# CIM → DCAT-3 Catalog (Mock-friendly Notebook)

This notebook builds a **DCAT-3 catalog** for the Colorado Information Marketplace (CIM).
It runs **without credentials** by default via a mock dataset layer, but includes a placeholder to wire in real Socrata pulls later.

**How to use**
1. Run the **Config** cell (`USE_MOCK = True` by default).
2. Run **Fetch** (mock or real depending on `USE_MOCK`).
3. Run **Build DCAT-3** to write outputs to `OUTPUT_DIR`:
   - `dcat3-catalog.jsonld` (JSON-LD, DCAT-3)
   - `dcat3-catalog.ttl` (simple Turtle preview; not a full RDF serialization)

> Created: 2025-09-19 — Author: ChatGPT for Joe Comeaux


In [1]:
# --- Config ---
USE_MOCK = False            # Change to False when wiring real Socrata pulls
OUTPUT_DIR = "/mnt/data"   # Where to save outputs

# Catalog metadata
CATALOG_URI = "https://data.colorado.gov/catalog/cim-catalog"
CATALOG_TITLE = "Colorado Information Marketplace – DCAT-3 Catalog"
CATALOG_DESCRIPTION = "DCAT-3 catalog built from CIM datasets. Set USE_MOCK=True to run without credentials."
PUBLISHER_NAME = "Business Intelligence Center (BIC)"
PUBLISHER_HOMEPAGE = "https://bic.colorado.gov/"

# Optional Socrata placeholders (used only if USE_MOCK=False)
SOC_DOMAIN = "data.colorado.gov"
SOC_APP_TOKEN = None
SOC_USERNAME = None
SOC_PASSWORD = None
DATASET_IDS = []        # e.g., ["abcd-1234", "wxyz-5678"]
SOC_SEARCH_TAGS = []    # e.g., ["transportation","finance"]

In [ ]:
# --- Fetch CIM datasets ---
from pathlib import Path
import json, datetime

def mock_fetch_cim_datasets():
    """Return a minimal but realistic set of dataset dicts for DCAT-3 assembly."""
    return [
        {
            "identifier": "abcd-1234",
            "title": "Colorado State Agencies",
            "description": "Directory of Colorado state agencies with points of contact and websites.",
            "keywords": ["Colorado", "agencies", "directory", "contacts"],
            "publisher": {"name": "Business Intelligence Center (BIC)", "homepage": "https://bic.colorado.gov/"},
            "contactPoint": {"fn": "CIM Support", "hasEmail": "mailto:cim@state.co.us"},
            "issued": "2020-05-15",
            "modified": "2025-08-30",
            "accrualPeriodicity": "irregular",
            "spatial": "Colorado",
            "temporal": {"startDate": "2020-01-01", "endDate": "2025-08-30"},
            "landingPage": "https://data.colorado.gov/Organizations/Colorado-State-Agencies/abcd-1234",
            "distributions": [
                {
                    "title": "Download CSV",
                    "description": "CSV export of agency directory",
                    "format": "text/csv",
                    "accessURL": "https://data.colorado.gov/resource/abcd-1234.csv",
                    "downloadURL": "https://data.colorado.gov/api/views/abcd-1234/rows.csv?accessType=DOWNLOAD",
                },
                {
                    "title": "Socrata API (SODA)",
                    "description": "Programmatic access via SODA API",
                    "format": "application/json",
                    "accessURL": "https://data.colorado.gov/resource/abcd-1234.json",
                },
            ],
        },
        {
            "identifier": "wxyz-5678",
            "title": "Colorado Counties Boundaries",
            "description": "Geospatial polygon boundaries for all Colorado counties.",
            "keywords": ["Colorado", "counties", "boundaries", "GIS", "geospatial"],
            "publisher": {"name": "Colorado Information Marketplace", "homepage": "https://data.colorado.gov/"},
            "contactPoint": {"fn": "CIM GIS Team", "hasEmail": "mailto:cim-gis@state.co.us"},
            "issued": "2018-03-01",
            "modified": "2025-07-12",
            "accrualPeriodicity": "asNeeded",
            "spatial": "Colorado",
            "temporal": {"startDate": "2018-03-01", "endDate": "2025-07-12"},
            "landingPage": "https://data.colorado.gov/Geospatial/Colorado-Counties/wxyz-5678",
            "distributions": [
                {
                    "title": "GeoJSON",
                    "description": "GeoJSON download",
                    "format": "application/geo+json",
                    "accessURL": "https://data.colorado.gov/resource/wxyz-5678.geojson",
                    "downloadURL": "https://data.colorado.gov/api/geospatial/wxyz-5678?method=export&format=GeoJSON",
                },
                {
                    "title": "Shapefile (ZIP)",
                    "description": "Shapefile export in ZIP",
                    "format": "application/zip",
                    "accessURL": "https://data.colorado.gov/api/geospatial/wxyz-5678?method=export&format=Shapefile",
                },
            ],
        },
    ]

def fetch_cim_datasets_socrata():
    """Placeholder for real Socrata pulls.
    Suggested approach:
      1) Query Catalog API: https://{SOC_DOMAIN}/api/catalog/v1
      2) For each result, call Views API: https://{SOC_DOMAIN}/api/views/{4x4-id}
      3) Map fields into the dicts returned by mock_fetch_cim_datasets().
    """
    raise NotImplementedError("Wire this to Socrata and map fields to the DS schema used below.")

# Choose source
if USE_MOCK:
    datasets = mock_fetch_cim_datasets()
else:
    datasets = fetch_cim_datasets_socrata()

print(f"Loaded {len(datasets)} dataset(s). Example titles:")
for ds in datasets:
    print(" -", ds["title"])

In [ ]:
# --- Build DCAT-3 JSON-LD and Turtle preview ---
import json, datetime
from pathlib import Path

context = {
    "@vocab": "http://www.w3.org/ns/dcat#",
    "dcat": "http://www.w3.org/ns/dcat#",
    "dct": "http://purl.org/dc/terms/",
    "foaf": "http://xmlns.com/foaf/0.1/",
    "xsd": "http://www.w3.org/2001/XMLSchema#",
    "schema": "http://schema.org/",
    "id": "@id",
    "type": "@type",
    "title": "dct:title",
    "description": "dct:description",
    "keyword": "dcat:keyword",
    "dataset": "dcat:dataset",
    "distribution": "dcat:distribution",
    "publisher": "dct:publisher",
    "contactPoint": "dcat:contactPoint",
    "landingPage": "dcat:landingPage",
    "identifier": "dct:identifier",
    "issued": {"@id": "dct:issued", "@type": "xsd:date"},
    "modified": {"@id": "dct:modified", "@type": "xsd:date"},
    "accrualPeriodicity": "dct:accrualPeriodicity",
    "spatial": "dct:spatial",
    "temporal": "dct:temporal",
    "startDate": {"@id": "schema:startDate", "@type": "xsd:date"},
    "endDate": {"@id": "schema:endDate", "@type": "xsd:date"},
    "accessURL": {"@id": "dcat:accessURL", "@type": "@id"},
    "downloadURL": {"@id": "dcat:downloadURL", "@type": "@id"},
    "format": "dct:format",
    "foaf_name": "foaf:name",
    "foaf_homepage": {"@id": "foaf:homepage", "@type": "@id"},
    "vcard_fn": "schema:name",
    "vcard_hasEmail": {"@id": "schema:email", "@type": "@id"}
}

now_date = datetime.date.today().isoformat()

catalog = {
    "@context": context,
    "id": CATALOG_URI,
    "type": "Catalog",
    "title": CATALOG_TITLE,
    "description": CATALOG_DESCRIPTION,
    "modified": now_date,
    "publisher": {
        "type": "foaf:Organization",
        "foaf_name": PUBLISHER_NAME,
        "foaf_homepage": PUBLISHER_HOMEPAGE,
    },
    "dataset": [],
}

for ds in datasets:
    ds_node = {
        "type": "Dataset",
        "identifier": ds["identifier"],
        "title": ds["title"],
        "description": ds["description"],
        "keyword": ds.get("keywords", []),
        "landingPage": ds.get("landingPage"),
        "issued": ds.get("issued"),
        "modified": ds.get("modified"),
        "accrualPeriodicity": ds.get("accrualPeriodicity"),
        "spatial": ds.get("spatial"),
        "temporal": {
            "type": "dct:PeriodOfTime",
            "startDate": ds.get("temporal", {}).get("startDate"),
            "endDate": ds.get("temporal", {}).get("endDate"),
        },
        "publisher": {
            "type": "foaf:Organization",
            "foaf_name": ds["publisher"]["name"],
            "foaf_homepage": ds["publisher"]["homepage"],
        },
        "contactPoint": {
            "type": "schema:ContactPoint",
            "vcard_fn": ds["contactPoint"]["fn"],
            "vcard_hasEmail": ds["contactPoint"]["hasEmail"],
        },
        "distribution": [],
    }
    for dist in ds.get("distributions", []):
        dnode = {
            "type": "Distribution",
            "title": dist.get("title"),
            "description": dist.get("description"),
            "format": dist.get("format"),
            "accessURL": dist.get("accessURL"),
        }
        if dist.get("downloadURL"):
            dnode["downloadURL"] = dist["downloadURL"]
        ds_node["distribution"].append(dnode)
    catalog["dataset"].append(ds_node)

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
jsonld_path = out_dir / "dcat3-catalog.jsonld"
ttl_path = out_dir / "dcat3-catalog.ttl"

with open(jsonld_path, "w", encoding="utf-8") as f:
    json.dump(catalog, f, indent=2, ensure_ascii=False)

# Simple Turtle preview (not a full RDF serialization)
ttl_lines = [
    "@prefix dcat: <http://www.w3.org/ns/dcat#> .",
    "@prefix dct: <http://purl.org/dc/terms/> .",
    "@prefix foaf: <http://xmlns.com/foaf/0.1/> .",
    f"<{CATALOG_URI}> a dcat:Catalog ;",
    f'  dct:title "{CATALOG_TITLE}" ;',
    f'  dct:description "{CATALOG_DESCRIPTION}" ;',
    f'  dct:modified "{now_date}" ;',
    f'  dct:publisher [ a foaf:Organization ; foaf:name "{PUBLISHER_NAME}" ; foaf:homepage <{PUBLISHER_HOMEPAGE}> ] ;'
]

for ds in datasets:
    ds_uri = f"{CATALOG_URI}#dataset-{ds['identifier']}"
    ttl_lines.append(f"  dcat:dataset <{ds_uri}> ;")
ttl_lines[-1] = ttl_lines[-1].rstrip(" ;") + " .\n"

for ds in datasets:
    ds_uri = f"{CATALOG_URI}#dataset-{ds['identifier']}"
    ttl_lines += [
        f"<{ds_uri}> a dcat:Dataset ;",
        f'  dct:identifier "{ds["identifier"]}" ;',
        f'  dct:title "{ds["title"]}" ;',
        f'  dct:description "{ds["description"]}" ;',
        f'  dct:issued "{ds.get("issued","")}" ;',
        f'  dct:modified "{ds.get("modified","")}" ;',
        f'  dct:spatial "{ds.get("spatial","")}" ;',
        f'  dct:publisher [ a foaf:Organization ; foaf:name "{ds["publisher"]["name"]}" ; foaf:homepage <{ds["publisher"]["homepage"]}> ] ;',
    ]
    for i, dist in enumerate(ds.get("distributions", []), start=1):
        dist_uri = f"{ds_uri}#dist-{i}"
        ttl_lines.append(f"  dcat:distribution <{dist_uri}> ;")
    ttl_lines[-1] = ttl_lines[-1].rstrip(" ;") + " ."
    for i, dist in enumerate(ds.get("distributions", []), start=1):
        dist_uri = f"{ds_uri}#dist-{i}"
        ttl_lines += [
            f"<{dist_uri}> a dcat:Distribution ;",
            f'  dct:title "{dist.get("title","")}" ;',
            f'  dct:description "{dist.get("description","")}" ;',
            f"  dct:format \"{dist.get('format','')}\" ;",
            f"  dcat:accessURL <{dist.get('accessURL','')}> ",
        ]
        if dist.get("downloadURL"):
            ttl_lines[-1] = ttl_lines[-1] + f";\n  dcat:downloadURL <{dist['downloadURL']}> .\n"
        else:
            ttl_lines[-1] = ttl_lines[-1] + ".\n"

ttl_path.write_text("\n".join(ttl_lines), encoding="utf-8")

print("Notebook will be saved to:", out_path)
# Save notebook
out_path.write_text(nbf.writes(nb), encoding="utf-8")

print("\nCreated files in /mnt/data/:")
for p in sorted(Path("/mnt/data").glob("*")):
    print(" -", p.name)